# 04 — Analysis

Check experiment completeness, compute grouped bootstrap intervals and all stored diagnostics, and save tables and figures. Complete notebook 03 first. This notebook never contacts Ollama.

In [ ]:
from pathlib import Path

import pandas as pd

from rubric_order.api import (
    analyze_experiment,
    experiment_completeness,
    experiment_directory,
    find_project_root,
    load_experiment_config,
    metric_tables,
)

PROJECT_ROOT = find_project_root(Path.cwd())

In [ ]:
DATA_PATH = PROJECT_ROOT / "HelpSteer2"
CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot.yaml"
RUN_ID = "pilot-001"
RUNTIME_OVERRIDES = {}
ALLOW_PARTIAL_RESULTS = False
EXECUTE_ANALYSIS = False

In [ ]:
config = load_experiment_config(CONFIG_PATH, DATA_PATH, overrides=RUNTIME_OVERRIDES)
completeness = experiment_completeness(config, RUN_ID)
{
    key: completeness[key]
    for key in [
        "complete",
        "expected_rubrics",
        "valid_rubrics",
        "missing_rubrics",
        "invalid_rubrics",
        "expected_predictions",
        "completed_predictions",
        "missing_prediction_count",
    ]
}

In [ ]:
if completeness["missing_prediction_groups"]:
    display(pd.DataFrame(completeness["missing_prediction_groups"]))
else:
    print("No prediction artifacts are missing.")

In [ ]:
if EXECUTE_ANALYSIS and (completeness["complete"] or ALLOW_PARTIAL_RESULTS):
    analysis = analyze_experiment(config, RUN_ID, allow_partial_results=ALLOW_PARTIAL_RESULTS)
    print(
        "Analysis complete. Tables and figures were saved under",
        experiment_directory(config, RUN_ID),
    )
else:
    analysis = None
    print(
        "Analysis not run: complete the artifacts listed above, or explicitly set "
        "EXECUTE_ANALYSIS = True only after completeness passes."
    )

In [ ]:
if analysis is not None:
    tables = metric_tables(config, RUN_ID)
    for name in [
        "primary_metrics",
        "agreement_metrics",
        "paired_differences",
        "repeat_control",
        "rubric_text",
        "rubric_similarity",
    ]:
        if tables.get(name):
            print(name)
            display(pd.DataFrame(tables[name]))

In [ ]:
if analysis is not None:
    for method, values in analysis["primary"].items():
        instability = values["calibration_order_instability"]
        change = values["score_change_rate"]
        print(
            f"{method}: instability={instability['estimate']:.3f} "
            f"(95% CI {instability['lower']:.3f}–{instability['upper']:.3f}), "
            f"score-change rate={change['estimate']:.3f}."
        )
    paired = analysis.get("paired_sequential_minus_batch", {})
    if paired:
        print(
            "Positive sequential-minus-batch instability means greater order dependence "
            "under sequential updating."
        )

In [ ]:
if analysis is not None:
    figure_dir = experiment_directory(config, RUN_ID) / "figures"
    [str(figure_dir / name) for name in analysis.get("plots", [])]